# Uygulama 2 — Mitokondri Farklı Konuşur: COX1 Geni

**Hafta 2** — Seq nesnesi, transkripsiyon, translasyon, genetik kod tabloları

**Veri:** NC_012920.1 (FASTA) — COX1: 5904..7445

---

**Senaryo:** Bir DNA barkodlama projesinde türleri ayırt etmek için mitokondriyal COX1 geni kullanılıyor. Ekip arkadaşın insan COX1 dizisini Biopython'un **varsayılan** ayarıyla proteine çevirmiş ve *"Bu gen bozuk, protein 6. amino asitte bitiyor!"* demiş.

Gerçekten bir sorun var mı? Yoksa ekip arkadaşın bir hata mı yapıyor?

> **İpucu:** Mitokondri, hücrenin geri kalanından farklı bir genetik kod kullanır!

## Arka Plan — Merkezi Dogma ve Genetik Kod

DNA'dan proteine giden yol:

```
DNA  →(transkripsiyon)→  mRNA  →(translasyon)→  Protein
T→U                       3 harf = 1 kodon = 1 amino asit
```

- **Transkripsiyon:** DNA'daki T'ler U'ya dönüşür (DNA → mRNA)
- **Translasyon:** Her 3 harflik grup (kodon) bir amino aside karşılık gelir
- **Dur kodonu:** UAA, UAG, UGA → "protein burada biter" sinyali

### Peki sorun ne?

Standart genetik kodda **TGA = dur** (protein biter). Ama mitokondride **TGA = Triptofan (Trp/W)** — yani protein devam eder! Ekip arkadaşın yanlış tabloyu kullanmış.

## Kalem-Kağıt Isınması

COX1'in ilk 21 bazı: `ATG TTC GCC GAC CGT TGA CTA`

**(a) mRNA dizisi** (T → U):
- `AUG UUC GCC GAC CGU UGA CUA`

**(b) Standart genetik kodla (tablo 1) çeviri:**

| Kodon | Amino Asit |
|-------|------------|
| AUG | Met (M) — başlangıç |
| UUC | Phe (F) |
| GCC | Ala (A) |
| GAC | Asp (D) |
| CGU | Arg (R) |
| **UGA** | **\* (DUR!)** |
| CUA | çevrilmez |

Sonuç: `MFADR*` → 6. pozisyonda bitiyor! Ekip arkadaşının gördüğü bu.

**(c) Omurgalı mitokondriyal koduyla (tablo 2) çeviri:**

| Kodon | Standart | Mitokondri |
|-------|----------|------------|
| UGA | DUR (*) | **Trp (W)** |
| AUA | Ile (I) | **Met (M)** |
| AGA/AGG | Arg (R) | **DUR (\*)** |

Sonuç: `MFADRWL...` → protein devam ediyor!

---
## Görev 0 — Kütüphaneler ve Veriyi Yükle

Uygulama 1'de indirdiğimiz FASTA dosyasını kullanıyoruz. Eğer dosya yoksa NCBI'dan tekrar indiriyoruz.

In [ ]:
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.Data import CodonTable
import os

Entrez.email = "semra@example.com"

# Önce yerel dosyayı dene, yoksa indir
if os.path.exists("NC_012920.fasta"):
    record = SeqIO.read("NC_012920.fasta", "fasta")
    print("Yerel dosyadan okundu.")
else:
    handle = Entrez.efetch(db="nucleotide", id="NC_012920.1",
                           rettype="fasta", retmode="text")
    record = SeqIO.read(handle, "fasta")
    handle.close()
    SeqIO.write(record, "NC_012920.fasta", "fasta")
    print("NCBI'dan indirildi ve kaydedildi.")

genom = record.seq.upper()
print(f"Genom uzunluğu: {len(genom)} bp")

---
## Görev 1 — COX1 Bölgesini Kes

NCBI kaydına göre COX1 geni genomda **5904..7445** konumlarında (1-tabanlı, dahil).

Python'da dilimleme **0-tabanlı** ve **yarı açık** `[start:stop)`:
- 1-tabanlı 5904 → 0-tabanlı **5903**
- 1-tabanlı 7445 dahil → yarı açık **7445**
- Yani: `genom[5903:7445]`

In [ ]:
cox1 = genom[5903:7445]

print(f"COX1 uzunluğu: {len(cox1)} baz")
print(f"Kodon sayısı : {len(cox1) // 3} kodon (+ {len(cox1) % 3} kalan baz)")
print(f"İlk 21 baz   : {cox1[:21]}")
print(f"Son 6 baz     : {cox1[-6:]}")
print(f"\n3'ün katı mı? → {len(cox1) % 3 == 0}")

**Beklenen:** 1542 baz = 514 kodon, 3'ün tam katı.

---
## Görev 2 — Transkripsiyon (DNA → mRNA)

Transkripsiyon basitçe T → U değişimi.

Biopython'da `seq.transcribe()` bunu yapar.

In [ ]:
mrna = cox1.transcribe()

print(f"İlk 21 baz (DNA) : {cox1[:21]}")
print(f"İlk 21 baz (mRNA): {mrna[:21]}")
print(f"\nFark sadece T→U dönüşümü!")

---
## Görev 3 — Standart Kodla Çeviri (Tablo 1)

Biopython'da `seq.translate()` varsayılan olarak **standart genetik kodu** (tablo 1) kullanır.

- `*` = dur kodonu (protein burada biter)
- Ekip arkadaşının yaptığı tam olarak bu

In [ ]:
protein_standart = cox1.translate(table=1)

# Dur kodonlarını say ve yerlerini bul
dur_sayisi = protein_standart.count('*')
ilk_dur = str(protein_standart).index('*') + 1  # 1-tabanlı

print(f"Protein (ilk 30 aa): {protein_standart[:30]}")
print(f"Toplam dur kodonu (*): {dur_sayisi}")
print(f"İlk dur kodonu pozisyonu: {ilk_dur}. amino asit")
print(f"\nEkip arkadaşı haklı gibi görünüyor... ama değil!")
print(f"Çünkü YANLIŞ tabloyu kullandı.")

---
## Görev 4 — Mitokondriyal Kodla Çeviri (Tablo 2)

Omurgalı mitokondriyal genetik kod = **tablo 2**

Standarttan farkları:
- **TGA → Trp (W)** (standartta dur)
- **ATA → Met (M)** (standartta Ile)
- **AGA/AGG → dur** (standartta Arg)

`cds=True` parametresi:
- Başlangıç kodonunu kontrol eder
- İlk amino asidi M yapar
- Son kodonun dur olmasını bekler

In [ ]:
# Tablo 2 ile çeviri
protein_mito = cox1.translate(table=2)

dur_sayisi_mito = protein_mito.count('*')

print(f"Protein (ilk 30 aa): {protein_mito[:30]}")
print(f"Toplam dur kodonu (*): {dur_sayisi_mito}")
print(f"Protein uzunluğu: {len(protein_mito)} aa")

# Dur kodonları nerede?
if dur_sayisi_mito > 0:
    durlar = [i+1 for i, aa in enumerate(protein_mito) if aa == '*']
    print(f"Dur kodonları pozisyonları: {durlar}")

In [ ]:
# cds=True ile çeviri — hata veriyor mu?
try:
    protein_cds = cox1.translate(table=2, cds=True)
    print(f"cds=True başarılı!")
    print(f"Protein (ilk 30 aa): {protein_cds[:30]}")
    print(f"Protein uzunluğu: {len(protein_cds)} aa")
except Exception as e:
    print(f"cds=True HATA verdi: {e}")

**Karşılaştırma:**
- Standart kod → protein 6. amino asitte bitiyor (yanlış!)
- Mito kodu → tam uzunlukta protein, doğru çeviri

`cds=True` ile çeviri başarılı olmalı çünkü gen ATG ile başlıyor ve uygun bir dur kodonuyla bitiyor.

---
## Görev 5 — Kritik Kodonların Sayısı

İki tablo arasında anlam değişen kodonlar:

| Kodon | Standart (Tablo 1) | Mitokondri (Tablo 2) |
|-------|-------------------|---------------------|
| TGA | DUR (*) | Trp (W) |
| ATA | Ile (I) | Met (M) |
| AGA | Arg (R) | DUR (*) |

Bu kodonları COX1'de sayalım — her biri kaç kez geçiyor?

In [ ]:
# Kodonlara ayır
kodonlar = [str(cox1[i:i+3]) for i in range(0, len(cox1) - 2, 3)]

kritik = {"TGA": 0, "ATA": 0, "AGA": 0, "AGG": 0}
for kodon in kodonlar:
    if kodon in kritik:
        kritik[kodon] += 1

print("Kodon   Sayı   Standart    Mitokondri")
print("-" * 45)
print(f"TGA     {kritik['TGA']:>3}    DUR (*)     Trp (W)")
print(f"ATA     {kritik['ATA']:>3}    Ile (I)     Met (M)")
print(f"AGA     {kritik['AGA']:>3}    Arg (R)     DUR (*)")
print(f"AGG     {kritik['AGG']:>3}    Arg (R)     DUR (*)")

print(f"\nEkip arkadaşına ne söylersin?")
print(f"→ 'Gen bozuk değil! Mitokondri farklı bir genetik kod kullanıyor.'")
print(f"→ 'Standartta dur anlamına gelen TGA, mitokondride Trp amino asidi.'")
print(f"→ 'translate(table=2) kullanman lazım, table=1 değil.'")

---
## Bonus — İki Genetik Kod Tablosunu Karşılaştır

Biopython tüm genetik kod tablolarını `Bio.Data.CodonTable` modülünde tutar. Aralarındaki farkları görelim.

In [ ]:
standart_tablo = CodonTable.unambiguous_dna_by_id[1]
mito_tablo = CodonTable.unambiguous_dna_by_id[2]

print(f"Tablo 1: {standart_tablo.names}")
print(f"Tablo 2: {mito_tablo.names}")
print()

# Farklı çevrilen kodonları bul
print("Kodon   Tablo 1          Tablo 2")
print("-" * 45)
for kodon in standart_tablo.forward_table:
    aa1 = standart_tablo.forward_table.get(kodon, '*')
    aa2 = mito_tablo.forward_table.get(kodon, '*')
    if aa1 != aa2:
        print(f"{kodon}     {aa1:15}  {aa2}")

# Dur kodonları farkı
print(f"\nDur kodonları (Tablo 1): {standart_tablo.stop_codons}")
print(f"Dur kodonları (Tablo 2): {mito_tablo.stop_codons}")

---

## Uygulama 2 Tamamlandı!

**Öğrendiğin kavramlar:**
- 1-tabanlı biyolojik koordinat → 0-tabanlı Python dilimi dönüşümü
- `seq.transcribe()` → DNA'dan mRNA'ya (T→U)
- `seq.translate(table=N)` → kodonları amino aside çevirme
- Standart (tablo 1) vs mitokondriyal (tablo 2) genetik kod farkı
- `cds=True` parametresinin ne yaptığı
- Neden doğru genetik kodu seçmek kritik

**Sonraki:** Uygulama 3 — LHON Mutasyonu (MutableSeq, nokta mutasyonu, sessiz/anlamlı değişim)